# Cleaning Malaysia's District-Level Crime Data for Tableau

This notebook turns the raw **Crimes by District & Crime Type** dataset (`crime_district`) from [data.gov.my](https://open.dosm.gov.my/data-catalogue/crime_district) into a tidy file that can be loaded straight into Tableau (or any BI tool) without double-counting.

**Source:** Royal Malaysia Police (PDRM), published by the Department of Statistics Malaysia (DOSM) under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

## What this notebook does

| Step | Purpose |
|---|---|
| 1. Setup | Configure paths and the cleaning rules in one place |
| 2. Load | Read the raw file (local copy, or download it if missing) |
| 3. Inspect | Look at the structure and the values in each column |
| 4. Basic quality checks | Missing values, duplicates, negative counts, unexpected categories |
| 5. Check the built-in totals | Find where the pre-summed rows disagree with the detail rows |
| 6. Remove total rows | Keep only the most granular rows so Tableau can't double-count |
| 7. Harmonise district names | Merge districts that were renamed partway through the series |
| 8. Add readable labels | Human-friendly crime names, category names and map-friendly state names |
| 9. Validate | Prove the cleaned file still reproduces the official national totals |
| 10. Export | Write the clean CSV and a data dictionary |

Every rule that changes the data is defined in **Step 1**, so if you reuse this notebook on a newer release of the dataset you only need to review that one cell. Each later step either applies those rules or checks that they still hold, and the notebook stops with an error if something unexpected appears.

## A note on what the numbers mean

The catalogue describes `crimes` as the number of actual crimes where a conviction was secured, but also says unreported crimes are excluded and refers to the "reported crime rate". The national totals here (e.g. 52,444 in 2023) are in line with PDRM's published index of *reported* crimes. Until DOSM clarifies, this project labels the measure **"recorded crimes"**.

## 1. Setup

All paths and cleaning rules live here.

In [1]:
from pathlib import Path
from difflib import SequenceMatcher
import pandas as pd

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

# ---- Paths -------------------------------------------------------------
RAW_PATH       = Path("data/raw/crime_district.csv")
CLEAN_PATH     = Path("data/processed/crime_district_clean.csv")
DICT_PATH      = Path("data/processed/data_dictionary.csv")
SOURCE_URL     = "https://storage.data.gov.my/publicsafety/crime_district.parquet"

# ---- Values the raw file uses to mark pre-summed (total) rows -----------
TOTAL_STATE    = "Malaysia"
TOTAL_DISTRICT = "All"
TOTAL_TYPE     = "all"

# ---- Districts renamed during the series: {old name: new name} ----------
# Keyed by (state, old_name) so a rename in one state can't affect another.
DISTRICT_RENAMES = {
    ("Kedah",    "Bandar Bharu"):     "Bandar Baharu",
    ("Pahang",   "Cameron Highland"): "Cameron Highlands",
    ("Selangor", "Sg. Buloh"):        "Sungai Buloh",
}

# ---- Readable labels -----------------------------------------------------
CATEGORY_LABELS = {
    "assault":  "Violent",
    "property": "Property",
}

TYPE_LABELS = {
    "causing_injury":           "Causing injury",
    "murder":                   "Murder",
    "rape":                     "Rape",
    "robbery_gang_armed":       "Robbery (gang, armed)",
    "robbery_gang_unarmed":     "Robbery (gang, unarmed)",
    "robbery_solo_armed":       "Robbery (solo, armed)",
    "robbery_solo_unarmed":     "Robbery (solo, unarmed)",
    "break_in":                 "Break-in",
    "theft_other":              "Theft (other)",
    "theft_vehicle_lorry":      "Vehicle theft (lorry/van)",
    "theft_vehicle_motorcar":   "Vehicle theft (car)",
    "theft_vehicle_motorcycle": "Vehicle theft (motorcycle)",
}

# State names as Tableau's built-in geocoding is expected to recognise them.
# Anything not listed keeps its original name. If a state shows up as
# "unknown" on your Tableau map, add or adjust its entry here.
STATE_MAP_NAMES = {
    "W.P. Kuala Lumpur": "Kuala Lumpur",
    "Pulau Pinang":      "Penang",
}

## 2. Load

The notebook uses the local copy in `data/raw/`. If it isn't there (e.g. on a fresh clone), it downloads the latest version from data.gov.my and saves it.

In [2]:
if RAW_PATH.exists():
    raw = pd.read_csv(RAW_PATH)
    print(f"Loaded local file: {RAW_PATH}")
else:
    print(f"{RAW_PATH} not found, downloading from {SOURCE_URL}")
    raw = pd.read_parquet(SOURCE_URL)  # needs pyarrow
    RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
    raw.to_csv(RAW_PATH, index=False)

print(f"{len(raw):,} rows x {raw.shape[1]} columns")
raw.head()

Loaded local file: data/raw/crime_district.csv
19,152 rows x 6 columns


,state,district,category,type,date,crimes
0,Malaysia,All,assault,all,2016-01-01,22327
1,Malaysia,All,assault,all,2017-01-01,21366
2,Malaysia,All,assault,all,2018-01-01,16902
3,Malaysia,All,assault,all,2019-01-01,16489
4,Malaysia,All,assault,all,2020-01-01,13279


## 3. Inspect

The data is in long format: one row per state, police district, crime type and year.

In [3]:
raw.dtypes

state         str
district      str
category      str
type          str
date          str
crimes      int64
dtype: object

In [4]:
for col in ["state", "district", "category", "type", "date"]:
    vals = sorted(raw[col].astype(str).unique())
    preview = vals if len(vals) <= 20 else vals[:20] + ["..."]
    print(f"{col}: {len(vals)} unique values\n   {preview}\n")

state: 15 unique values
   ['Johor', 'Kedah', 'Kelantan', 'Malaysia', 'Melaka', 'Negeri Sembilan', 'Pahang', 'Perak', 'Perlis', 'Pulau Pinang', 'Sabah', 'Sarawak', 'Selangor', 'Terengganu', 'W.P. Kuala Lumpur']

district: 160 unique values
   ['All', 'Alor Gajah', 'Ampang Jaya', 'Arau', 'Bachok', 'Baling', 'Bandar Baharu', 'Bandar Bharu', 'Barat Daya', 'Batu Gajah', 'Batu Pahat', 'Bau', 'Beaufort', 'Belaga', 'Beluran', 'Bentong', 'Bera', 'Besut', 'Betong', 'Bintulu', '...']

category: 2 unique values
   ['assault', 'property']

type: 13 unique values
   ['all', 'break_in', 'causing_injury', 'murder', 'rape', 'robbery_gang_armed', 'robbery_gang_unarmed', 'robbery_solo_armed', 'robbery_solo_unarmed', 'theft_other', 'theft_vehicle_lorry', 'theft_vehicle_motorcar', 'theft_vehicle_motorcycle']

date: 8 unique values
   ['2016-01-01', '2017-01-01', '2018-01-01', '2019-01-01', '2020-01-01', '2021-01-01', '2022-01-01', '2023-01-01']



Things to note:

- `date` is always 1 January, because the data is annual. We'll convert it to a plain integer `year`.
- `state = "Malaysia"`, `district = "All"` and `type = "all"` are **total rows**, not real places or crime types. Step 6 deals with these.
- W.P. Putrajaya and W.P. Labuan don't appear as separate states.

## 4. Basic quality checks

These checks stop the notebook if they fail, so problems in a future release can't slip through unnoticed.

In [5]:
key_cols = ["state", "district", "category", "type", "date"]

assert raw.isna().sum().sum() == 0, "Missing values found"
assert not raw.duplicated(key_cols).any(), "Duplicate rows for the same state/district/type/date"
assert (raw["crimes"] >= 0).all(), "Negative crime counts found"

unknown_categories = set(raw["category"]) - set(CATEGORY_LABELS)
unknown_types = set(raw["type"]) - set(TYPE_LABELS) - {TOTAL_TYPE}
assert not unknown_categories, f"New categories need labels in Step 1: {unknown_categories}"
assert not unknown_types, f"New crime types need labels in Step 1: {unknown_types}"

print("All basic checks passed.")
print(f"Zero-count rows: {(raw['crimes'] == 0).sum():,} (explicit zeros, kept as-is)")

All basic checks passed.
Zero-count rows: 4,001 (explicit zeros, kept as-is)


## 5. Check the built-in totals

The raw file includes three kinds of pre-summed rows. Before removing them, we check whether they agree with the detail rows underneath them. This tells us which numbers to trust.

In [6]:
is_detail = (
    (raw["state"] != TOTAL_STATE)
    & (raw["district"] != TOTAL_DISTRICT)
    & (raw["type"] != TOTAL_TYPE)
)
detail = raw[is_detail]

def compare(label, computed, official):
    """Show every row where the sum of detail rows differs from the published total."""
    both = pd.DataFrame({"sum_of_detail": computed, "published_total": official}).dropna()
    both["difference"] = both["published_total"] - both["sum_of_detail"]
    bad = both[both["difference"] != 0]
    print(f"{label}: {len(bad)} mismatches out of {len(both)}")
    return bad

# (a) District 'all' rows vs the sum of that district's crime types
district_all = raw[(raw["state"] != TOTAL_STATE) & (raw["district"] != TOTAL_DISTRICT) & (raw["type"] == TOTAL_TYPE)]
a = compare(
    "District 'all' rows vs sum of crime types",
    detail.groupby(["state", "district", "category", "date"])["crimes"].sum(),
    district_all.set_index(["state", "district", "category", "date"])["crimes"],
)

# (b) State 'All' rows vs the sum of the detail rows in that state
state_all = raw[(raw["state"] != TOTAL_STATE) & (raw["district"] == TOTAL_DISTRICT) & (raw["type"] != TOTAL_TYPE)]
b = compare(
    "State totals vs sum of district detail",
    detail.groupby(["state", "category", "type", "date"])["crimes"].sum(),
    state_all.set_index(["state", "category", "type", "date"])["crimes"],
)

# (c) National rows vs the sum of all detail rows
national = raw[(raw["state"] == TOTAL_STATE) & (raw["type"] != TOTAL_TYPE)]
c = compare(
    "National totals vs sum of all detail",
    detail.groupby(["category", "type", "date"])["crimes"].sum(),
    national.set_index(["category", "type", "date"])["crimes"],
)

a

District 'all' rows vs sum of crime types: 2 mismatches out of 2496
State totals vs sum of district detail: 0 mismatches out of 1344
National totals vs sum of all detail: 0 mismatches out of 96


sum_of_detail  published_total  difference
state    district category date                                                  
Selangor Serdang  property 2020-01-01            962             1282         320
                           2021-01-01            828             1197         369

**Finding.** The state and national totals match the detail rows exactly. The only disagreement is in the district-level `all` rows shown above (in the 2023 release: Serdang, Selangor, property crime, 2020 and 2021). Those `all` figures are higher than the sum of their own crime types, while the Selangor state total agrees with the detail rows.

**Decision.** Treat the detail rows as the source of truth and drop every total row. Tableau will rebuild any total it needs from the detail, so all totals in the dashboard will be consistent.

## 6. Remove total rows

After this step, every row is one specific crime type, in one police district, in one year. Summing any subset gives the right answer.

In [7]:
clean = raw[is_detail].copy()
print(f"Kept {len(clean):,} detail rows, dropped {len(raw) - len(clean):,} total rows.")

Kept 14,976 detail rows, dropped 4,176 total rows.


## 7. Harmonise district names

Some police districts change their spelling partway through the series. Left alone, Tableau would show them as two separate districts, each with half a trend line.

The cell below looks for likely renames automatically: pairs of districts in the same state whose names are similar and whose years never overlap. It then compares what it finds with the `DISTRICT_RENAMES` list in Step 1. On a new release, any new candidates are flagged for you to review.

In [8]:
years_by_district = clean.groupby(["state", "district"])["date"].apply(set)

candidates = []
for state, group in years_by_district.groupby(level="state"):
    names = group.index.get_level_values("district").tolist()
    for i, n1 in enumerate(names):
        for n2 in names[i + 1:]:
            similarity = SequenceMatcher(None, n1.lower(), n2.lower()).ratio()
            overlap = group[(state, n1)] & group[(state, n2)]
            if similarity >= 0.6 and not overlap:
                first, second = sorted([n1, n2], key=lambda n: min(group[(state, n)]))
                candidates.append({
                    "state": state,
                    "older_name": first,
                    "newer_name": second,
                    "older_years": f"{min(group[(state, first)])[:4]}–{max(group[(state, first)])[:4]}",
                    "newer_years": f"{min(group[(state, second)])[:4]}–{max(group[(state, second)])[:4]}",
                    "similarity": round(similarity, 2),
                    "in_rename_list": (state, first) in DISTRICT_RENAMES,
                })

candidates = pd.DataFrame(candidates)
candidates

,state,older_name,newer_name,older_years,newer_years,similarity,in_rename_list
0,Kedah,Bandar Bharu,Bandar Baharu,2016–2019,2020–2023,0.96,True
1,Pahang,Cameron Highland,Cameron Highlands,2016–2019,2020–2023,0.97,True
2,Selangor,Sg. Buloh,Sungai Buloh,2016–2020,2021–2023,0.76,True


In [9]:
unreviewed = candidates[~candidates["in_rename_list"]] if len(candidates) else candidates
if len(unreviewed):
    print("WARNING: possible renames not yet in DISTRICT_RENAMES. Review these and update Step 1:")
    display(unreviewed)
else:
    print("Every detected candidate is covered by DISTRICT_RENAMES.")

missing = [k for k in DISTRICT_RENAMES if k not in years_by_district.index]
assert not missing, f"DISTRICT_RENAMES refers to districts not in the data: {missing}"

Every detected candidate is covered by DISTRICT_RENAMES.


In [10]:
clean["district"] = [
    DISTRICT_RENAMES.get((s, d), d) for s, d in zip(clean["state"], clean["district"])
]

assert not clean.duplicated(key_cols).any(), "Merging districts created overlapping years"
print(f"Police districts after merging: {clean.groupby(['state', 'district']).ngroups}")

Police districts after merging: 156


## 8. Add readable labels and tidy columns

The original codes are kept alongside the labels, so the clean file can always be traced back to the raw data.

In [11]:
clean["year"] = pd.to_datetime(clean["date"]).dt.year
clean["category_label"] = clean["category"].map(CATEGORY_LABELS)
clean["type_label"] = clean["type"].map(TYPE_LABELS)
clean["state_map_name"] = clean["state"].replace(STATE_MAP_NAMES)

clean = (
    clean[["year", "state", "state_map_name", "district",
           "category", "category_label", "type", "type_label", "crimes"]]
    .sort_values(["year", "state", "district", "category", "type"])
    .reset_index(drop=True)
)

assert clean[["category_label", "type_label"]].notna().all().all()
clean.head(10)

,year,state,state_map_name,district,category,category_label,type,type_label,crimes
0,2016,Johor,Johor,Batu Pahat,assault,Violent,causing_injury,Causing injury,39
1,2016,Johor,Johor,Batu Pahat,assault,Violent,murder,Murder,6
2,2016,Johor,Johor,Batu Pahat,assault,Violent,rape,Rape,29
3,2016,Johor,Johor,Batu Pahat,assault,Violent,robbery_gang_armed,"Robbery (gang, armed)",1
4,2016,Johor,Johor,Batu Pahat,assault,Violent,robbery_gang_unarmed,"Robbery (gang, unarmed)",56
5,2016,Johor,Johor,Batu Pahat,assault,Violent,robbery_solo_armed,"Robbery (solo, armed)",1
6,2016,Johor,Johor,Batu Pahat,assault,Violent,robbery_solo_unarmed,"Robbery (solo, unarmed)",28
7,2016,Johor,Johor,Batu Pahat,property,Property,break_in,Break-in,115
8,2016,Johor,Johor,Batu Pahat,property,Property,theft_other,Theft (other),192
9,2016,Johor,Johor,Batu Pahat,property,Property,theft_vehicle_lorry,Vehicle theft (lorry/van),16


## 9. Validate

The final check: the cleaned file must still reproduce the official national totals published in the raw data, for every year, category and crime type. If this passes, nothing was lost or double-counted during cleaning.

In [12]:
official = (
    national.assign(year=pd.to_datetime(national["date"]).dt.year)
    .set_index(["year", "category", "type"])["crimes"]
    .sort_index()
)
rebuilt = clean.groupby(["year", "category", "type"])["crimes"].sum().sort_index()

pd.testing.assert_series_equal(rebuilt, official, check_names=False)
print("Cleaned data reproduces every official national total.")

summary = clean.pivot_table(index="year", columns="category_label", values="crimes", aggfunc="sum")
summary["Total"] = summary.sum(axis=1)
summary

Cleaned data reproduces every official national total.


category_label,Property,Violent,Total
year,,,
2016,90028,22327,112355
2017,77804,21366,99170
2018,71760,16902,88662
2019,66967,16489,83456
2020,52344,13279,65623
2021,41479,11495,52974
2022,40465,10348,50813
2023,41991,10453,52444


## 10. Export

Two files are written to `data/processed/`: the clean dataset and a data dictionary that describes each column.

In [13]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
clean.to_csv(CLEAN_PATH, index=False)

data_dictionary = pd.DataFrame([
    ("year",           "Integer",     "Calendar year of the record (annual data)."),
    ("state",          "Text",        "State as named in the source data (14 states; Putrajaya and Labuan are not reported separately)."),
    ("state_map_name", "Text",        "State name adjusted for Tableau's built-in map geocoding. Use this field for maps."),
    ("district",       "Text",        "PDRM police district (not the same as an administrative district). Renamed districts merged under their current name."),
    ("category",       "Text",        "Source code for the crime category: 'assault' or 'property'."),
    ("category_label", "Text",        "Readable crime category: 'Violent' or 'Property'."),
    ("type",           "Text",        "Source code for the specific crime type."),
    ("type_label",     "Text",        "Readable crime type."),
    ("crimes",         "Integer",     "Number of recorded crimes. Safe to sum across any combination of fields."),
], columns=["column", "type", "description"])
data_dictionary.to_csv(DICT_PATH, index=False)

print(f"Wrote {len(clean):,} rows to {CLEAN_PATH}")
print(f"Wrote data dictionary to {DICT_PATH}")
data_dictionary

Wrote 14,976 rows to data/processed/crime_district_clean.csv
Wrote data dictionary to data/processed/data_dictionary.csv


,column,type,description
0,year,Integer,Calendar year of the record (annual data).
1,state,Text,State as named in the source data (14 states; ...
2,state_map_name,Text,State name adjusted for Tableau's built-in map...
3,district,Text,PDRM police district (not the same as an admin...
4,category,Text,Source code for the crime category: 'assault' ...
5,category_label,Text,Readable crime category: 'Violent' or 'Property'.
6,type,Text,Source code for the specific crime type.
7,type_label,Text,Readable crime type.
8,crimes,Integer,Number of recorded crimes. Safe to sum across ...


## Summary of changes

| Change | Why |
|---|---|
| Dropped all `Malaysia`, `All` and `all` total rows | Prevents double-counting in Tableau; removes the inconsistent Serdang totals |
| Merged three renamed districts | Keeps each district's trend line continuous |
| Converted `date` to integer `year` | The data is annual; a year is simpler to filter on |
| Added readable labels next to source codes | Clean filter menus, while staying traceable to the raw data |
| Added `state_map_name` | Helps Tableau place states on its built-in map |

**Using the file in Tableau:** connect to `crime_district_clean.csv`, set `state_map_name` to the *State/Province* geographic role, and use `SUM([crimes])` everywhere. Because only detail rows remain, any combination of filters gives a correct total.